In [ ]:
from __future__ import annotations
import os

CPU_AFFINITY_ENABLED = False
CPU_START = 0
CPU_STOP  = 8  # exclusive

if CPU_AFFINITY_ENABLED:
    cpus = set(range(int(CPU_START), int(CPU_STOP)))
    if hasattr(os, "sched_setaffinity"):
        os.sched_setaffinity(0, cpus)
        print(f"Pinned to CPUs {min(cpus)}..{max(cpus)}")
    else:
        print("CPU affinity not supported on this platform.")
else:
    print("CPU affinity unchanged.")


# Lyapunov Campaign Characterization: $N_x = 20,\; N_y \in \{30,40,50\}$

Loads the completed `N20_Ny30-50_nsh1_a1-1_S25_cyclesNy` campaign (12 runs:
3 system sizes $\times$ 2 DW configurations $\times$ 2 protocols) and produces
Lyapunov, Chern, filling-fraction, and spatial charge-profile diagnostics.

The **Lyapunov gap proxy** is $\Delta_\lambda \equiv \min_i |\lambda_i|$
evaluated per sample and cycle. Each run uses $N_{\rm cyc} = N_y$ total Markov cycles.

**Notebook structure:**
- **Part 1** — Perfect correction: DW off (§1.1), DW on (§1.2), scaling summary (§1.3).
- **Part 2** — Post-selection: DW off (§2.1), DW on (§2.2), scaling summary (§2.3).


#### Imports and plot style

In [ ]:
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd

CAMPAIGN_ID = "N20_Ny30-50_nsh1_a1-1_S25_cyclesNy"
DPI = 300
W1  = 3.375   # single-panel width (in)
W2  = 6.75    # three-column / two-column panel width (in)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "colab_lyapunov" / "gpu_data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate project root containing colab_lyapunov/gpu_data")


PROJECT_ROOT  = find_project_root()
CAMPAIGN_ROOT = (
    PROJECT_ROOT / "colab_lyapunov" / "gpu_data"
    / "lyapunov_spectra" / "campaigns" / CAMPAIGN_ID
)

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_family = "CMU Sans Serif" if "CMU Sans Serif" in available_fonts else "DejaVu Sans"
mpl.rcParams.update({
    "font.family":     font_family,
    "font.size":       8,
    "axes.titlesize":  8,
    "axes.labelsize":  8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi":      DPI,
    "savefig.dpi":     DPI,
    "axes.linewidth":  0.7,
    "lines.linewidth": 1.2,
})

print(f"project root : {PROJECT_ROOT}")
print(f"campaign root: {CAMPAIGN_ROOT}")
print(f"font         : {font_family}")


#### Campaign config: system sizes, config-ID lists, colour map, row-cycle indices

In [ ]:
NY_VALUES = [30, 40, 50]
NX = 20

PERF_CORR_DW0 = [f"N20x{ny}_DW0_dwtrunc0_a2-1_nsh1_perfect_correction"  for ny in NY_VALUES]
PERF_CORR_DW1 = [f"N20x{ny}_DW1_dwtrunc1_a2-30_nsh1_perfect_correction" for ny in NY_VALUES]
POSTSEL_DW0   = [f"N20x{ny}_DW0_dwtrunc0_a2-1_nsh1_postselect"           for ny in NY_VALUES]
POSTSEL_DW1   = [f"N20x{ny}_DW1_dwtrunc1_a2-30_nsh1_postselect"          for ny in NY_VALUES]

ALL_CONFIGS = PERF_CORR_DW0 + PERF_CORR_DW1 + POSTSEL_DW0 + POSTSEL_DW1

_tab10 = mpl.colormaps["tab10"].colors
COLORS = {ny: _tab10[i] for i, ny in enumerate(NY_VALUES)}

ROW_CYCLES = [5, 10, 20, None]   # None = final cycle of each config


## Data Loading


#### Load all 12 runs from disk; compute filling fraction, x-profile, and $\min_i|\lambda_i|$

In [ ]:
run_index = pd.read_csv(CAMPAIGN_ROOT / "run_index.csv")
runs: dict = {}

for config_id in ALL_CONFIGS:
    run_dir = CAMPAIGN_ROOT / "runs" / config_id
    summary = json.loads((run_dir / "run_summary.json").read_text())
    with np.load(run_dir / "lyapunov_spectra.npz", allow_pickle=False) as d:
        spectra       = d["lyapunov_spectra"]
        normalization = str(d["normalization"].tolist())
    with np.load(run_dir / "real_space_chern.npz", allow_pickle=False) as d:
        chern  = d["real_space_chern"]
    with np.load(run_dir / "local_charge_cell.npz", allow_pickle=False) as d:
        charge = d["local_charge_cell"]
    nx_s, ny_s   = charge.shape[2], charge.shape[3]
    total_charge = charge.sum(axis=(-2, -1))
    runs[config_id] = {
        "summary":          summary,
        "spectra":          spectra,
        "chern":            chern,
        "charge":           charge,
        "normalization":    normalization,
        "total_charge":     total_charge,
        "filling_fraction": total_charge / float(nx_s * ny_s),
        "x_profile":        charge.mean(axis=-1),             # (S, cycles, Nx)
        "lyapunov_min_abs": np.min(np.abs(spectra), axis=-1), # (S, cycles)
    }

for cfg, run in runs.items():
    s = run["summary"]
    S, C, _ = run["spectra"].shape
    assert S == int(s["actual_samples"]),                  cfg
    assert C == int(s["cycles"]),                           cfg
    assert run["chern"].shape  == (S, C),                  cfg
    assert run["charge"].shape == (S, C, 20, int(s["Ny"])), cfg
    assert np.isfinite(run["spectra"]).all(),               cfg
    assert np.isfinite(run["chern"]).all(),                 cfg
    assert np.isfinite(run["charge"]).all(),                cfg

print("All 12 runs loaded and validated.")

summary_rows = []
for cfg in ALL_CONFIGS:
    s = runs[cfg]["summary"]
    summary_rows.append({
        "config":   cfg,
        "protocol": s["protocol"],
        "Ny":       s["Ny"],
        "DW":       s["DW"],
        "samples":  s["actual_samples"],
        "cycles":   s["cycles"],
        "n_vec":    runs[cfg]["spectra"].shape[2],
    })
pd.DataFrame(summary_rows)


#### Helper functions: `sample_mean_stderr`, `row_cycle`, `hist_or_vline`, `annotate_grid`, `cycle_axis`

In [ ]:
def sample_mean_stderr(a: np.ndarray, axis: int = 0):
    mean = np.mean(a, axis=axis)
    n    = a.shape[axis]
    err  = np.std(a, axis=axis, ddof=1) / np.sqrt(float(n)) if n > 1 else np.zeros_like(mean)
    return mean, err


def row_cycle(cfg: str, cyc) -> int:
    # None -> final cycle of cfg
    return runs[cfg]["summary"]["cycles"] if cyc is None else cyc


def hist_or_vline(ax, vals, bins, color, n_samples: int) -> None:
    # histogram for n>1 samples; vertical line for n=1
    if n_samples == 1:
        v = float(vals.flat[0])
        ax.axvline(v, color=color, lw=1.5)
        ax.plot(v, 0.5, marker="o", color=color, ms=3,
                transform=ax.get_xaxis_transform())
    else:
        ax.hist(vals, bins=bins, color=color, alpha=0.82, linewidth=0)


def annotate_grid(axes, cfgs) -> None:
    # add Ny column titles and cycle row labels to a 4x3 axes grid
    for col, cfg in enumerate(cfgs):
        axes[0, col].set_title(f"$N_y = {runs[cfg]['summary']['Ny']}$")
    for row, cyc in enumerate(ROW_CYCLES):
        axes[row, 0].set_ylabel("final" if cyc is None else f"cycle {cyc}")


def cycle_axis(cfg: str) -> np.ndarray:
    return np.arange(1, runs[cfg]["summary"]["cycles"] + 1)


## Part 1: Perfect Correction


### 1.1 Perfect Correction — DW = False


#### Lyapunov spectra histograms at selected cycles — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
_all_vals = np.concatenate([
    runs[c]["spectra"][:, row_cycle(c, cyc) - 1, :].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_bins = np.linspace(_all_vals.min(), _all_vals.max(), 81)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["spectra"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :].ravel()
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\lambda$")
annotate_grid(axes, _cfgs)
fig.suptitle("Lyapunov spectra — perfect correction, DW off", y=1.02)
plt.show()


#### Sample-averaged sorted Lyapunov spectrum — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny    = runs[cfg]["summary"]["Ny"]
    color = COLORS[ny]
    n_vec = runs[cfg]["spectra"].shape[2]
    frac  = np.arange(n_vec) / n_vec
    for row, cyc in enumerate(ROW_CYCLES):
        ax        = axes[row, col]
        mean_spec = np.sort(np.mean(runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :], axis=0))
        ax.plot(frac, mean_spec, color=color, lw=1.0)
        if row == 3:
            ax.set_xlabel(r"index $/ n_{\rm vec}$")
annotate_grid(axes, _cfgs)
fig.suptitle("Sample-averaged Lyapunov spectra — perfect correction, DW off", y=1.02)
plt.show()


#### Lyapunov gap proxy $\Delta_\lambda \equiv \min_i|\lambda_i|$ vs cycle — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["lyapunov_min_abs"], axis=0)
    ax.plot(cyc_ax, mean, color=color,
            label=f"$N_y={ny}$ (S={runs[cfg]['spectra'].shape[0]})")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\min_i |\lambda_i|$")
ax.set_yscale("log")
ax.set_title("Lyapunov gap proxy $\\Delta_\\lambda$ — perfect correction, DW off")
ax.legend(frameon=False)
plt.show()


#### Real-space Chern number histograms at selected cycles — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
_all_chern = np.concatenate([
    runs[c]["chern"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(1.0, float(_all_chern.max() - _all_chern.min()))
_bins = np.linspace(float(_all_chern.min()) - _pad, float(_all_chern.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["chern"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["chern"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$C$")
annotate_grid(axes, _cfgs)
fig.suptitle("Real-space Chern number histograms — perfect correction, DW off", y=1.02)
plt.show()


#### Sample-averaged Chern number $\langle\mathcal{C}\rangle$ vs cycle — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["chern"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$C$")
ax.set_title("Sample-averaged Chern number — perfect correction, DW off")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction histograms at selected cycles — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
_all_fill = np.concatenate([
    runs[c]["filling_fraction"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(0.01, float(_all_fill.max() - _all_fill.min()))
_bins = np.linspace(float(_all_fill.min()) - _pad, float(_all_fill.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["filling_fraction"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["filling_fraction"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
annotate_grid(axes, _cfgs)
fig.suptitle("Filling-fraction histograms — perfect correction, DW off", y=1.02)
plt.show()


#### Sample-averaged filling fraction $\nu$ vs cycle — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["filling_fraction"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
ax.set_title("Sample-averaged filling fraction — perfect correction, DW off")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction sample-to-sample std vs cycle — DW off, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    ff  = runs[cfg]["filling_fraction"]
    std = np.std(ff, axis=0, ddof=1) if ff.shape[0] > 1 else np.zeros(ff.shape[1])
    ax.plot(cyc_ax, std, color=color, label=f"$N_y={ny}$")
ax.set_xlabel("cycle")
ax.set_ylabel(r"std$_{\rm samples}(\nu)$")
ax.set_title("Filling-fraction sample std — perfect correction, DW off")
ax.legend(frameon=False)
plt.show()


#### Charge $x$-profile mean vs cycle — DW off, perfect correction

In [ ]:
_cfgs   = PERF_CORR_DW0
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]   # (S, cycles, n_x)
    mean, _ = sample_mean_stderr(xp, axis=0)
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, mean[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"$\langle N_y^{-1}\sum_y Q_{x,y}\rangle$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile mean — perfect correction, DW off", y=1.08)
plt.show()


#### Charge $x$-profile sample-to-sample std vs cycle — DW off, perfect correction

In [ ]:
_cfgs   = PERF_CORR_DW0
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]
    std    = np.std(xp, axis=0, ddof=1) if xp.shape[0] > 1 else np.zeros(xp.shape[1:])
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, std[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"std$_{\rm samples}(N_y^{-1}\sum_y Q_{x,y})$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile sample std — perfect correction, DW off", y=1.08)
plt.show()


### 1.2 Perfect Correction — DW = True


#### Lyapunov spectra histograms at selected cycles — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
_all_vals = np.concatenate([
    runs[c]["spectra"][:, row_cycle(c, cyc) - 1, :].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_bins = np.linspace(_all_vals.min(), _all_vals.max(), 81)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["spectra"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :].ravel()
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\lambda$")
annotate_grid(axes, _cfgs)
fig.suptitle("Lyapunov spectra — perfect correction, DW on", y=1.02)
plt.show()


#### Sample-averaged sorted Lyapunov spectrum — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny    = runs[cfg]["summary"]["Ny"]
    color = COLORS[ny]
    n_vec = runs[cfg]["spectra"].shape[2]
    frac  = np.arange(n_vec) / n_vec
    for row, cyc in enumerate(ROW_CYCLES):
        ax        = axes[row, col]
        mean_spec = np.sort(np.mean(runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :], axis=0))
        ax.plot(frac, mean_spec, color=color, lw=1.0)
        if row == 3:
            ax.set_xlabel(r"index $/ n_{\rm vec}$")
annotate_grid(axes, _cfgs)
fig.suptitle("Sample-averaged Lyapunov spectra — perfect correction, DW on", y=1.02)
plt.show()


#### Lyapunov gap proxy $\Delta_\lambda$ vs cycle — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["lyapunov_min_abs"], axis=0)
    ax.plot(cyc_ax, mean, color=color,
            label=f"$N_y={ny}$ (S={runs[cfg]['spectra'].shape[0]})")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\min_i |\lambda_i|$")
ax.set_yscale("log")
ax.set_title("Lyapunov gap proxy $\\Delta_\\lambda$ — perfect correction, DW on")
ax.legend(frameon=False)
plt.show()


#### Real-space Chern number histograms at selected cycles — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
_all_chern = np.concatenate([
    runs[c]["chern"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(1.0, float(_all_chern.max() - _all_chern.min()))
_bins = np.linspace(float(_all_chern.min()) - _pad, float(_all_chern.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["chern"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["chern"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$C$")
annotate_grid(axes, _cfgs)
fig.suptitle("Real-space Chern number histograms — perfect correction, DW on", y=1.02)
plt.show()


#### Sample-averaged Chern number $\langle\mathcal{C}\rangle$ vs cycle — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["chern"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$C$")
ax.set_title("Sample-averaged Chern number — perfect correction, DW on")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction histograms at selected cycles — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
_all_fill = np.concatenate([
    runs[c]["filling_fraction"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(0.01, float(_all_fill.max() - _all_fill.min()))
_bins = np.linspace(float(_all_fill.min()) - _pad, float(_all_fill.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["filling_fraction"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["filling_fraction"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
annotate_grid(axes, _cfgs)
fig.suptitle("Filling-fraction histograms — perfect correction, DW on", y=1.02)
plt.show()


#### Sample-averaged filling fraction $\nu$ vs cycle — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["filling_fraction"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
ax.set_title("Sample-averaged filling fraction — perfect correction, DW on")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction sample-to-sample std vs cycle — DW on, perfect correction

In [ ]:
_cfgs = PERF_CORR_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    ff  = runs[cfg]["filling_fraction"]
    std = np.std(ff, axis=0, ddof=1) if ff.shape[0] > 1 else np.zeros(ff.shape[1])
    ax.plot(cyc_ax, std, color=color, label=f"$N_y={ny}$")
ax.set_xlabel("cycle")
ax.set_ylabel(r"std$_{\rm samples}(\nu)$")
ax.set_title("Filling-fraction sample std — perfect correction, DW on")
ax.legend(frameon=False)
plt.show()


#### Charge $x$-profile mean vs cycle — DW on, perfect correction

In [ ]:
_cfgs   = PERF_CORR_DW1
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]   # (S, cycles, n_x)
    mean, _ = sample_mean_stderr(xp, axis=0)
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, mean[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"$\langle N_y^{-1}\sum_y Q_{x,y}\rangle$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile mean — perfect correction, DW on", y=1.08)
plt.show()


#### Charge $x$-profile sample-to-sample std vs cycle — DW on, perfect correction

In [ ]:
_cfgs   = PERF_CORR_DW1
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]
    std    = np.std(xp, axis=0, ddof=1) if xp.shape[0] > 1 else np.zeros(xp.shape[1:])
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, std[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"std$_{\rm samples}(N_y^{-1}\sum_y Q_{x,y})$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile sample std — perfect correction, DW on", y=1.08)
plt.show()


### 1.3 Summary: Lyapunov Gap Scaling (Perfect Correction)

The two figures below show how the Lyapunov gap proxy
$\Delta_\lambda \equiv \min_i|\lambda_i|$ depends on system size $N_y$
and evolves with cycle number under **perfect correction**.
Each subplot contrasts DW-off (left) and DW-on (right).


#### Final-cycle $\Delta_\lambda$ vs $N_y$ — perfect correction, DW off vs on

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W2, 2.0), constrained_layout=True)
for ax, cfgs, dw_label in zip(
    axes,
    [PERF_CORR_DW0, PERF_CORR_DW1],
    ["DW off", "DW on"],
):
    nys, means, errs = [], [], []
    for cfg in cfgs:
        ny      = runs[cfg]["summary"]["Ny"]
        final_i = runs[cfg]["summary"]["cycles"] - 1
        gap     = runs[cfg]["lyapunov_min_abs"][:, final_i]
        m, e    = sample_mean_stderr(gap, axis=0)
        nys.append(ny); means.append(float(m)); errs.append(float(e))
    nys = np.array(nys)
    ax.errorbar(nys, means, yerr=errs, fmt="o-", color="tab:blue", capsize=3, ms=5)
    ax.set_xlabel(r"$N_y$")
    ax.set_ylabel(r"$\Delta_\lambda$ (final cycle)")
    ax.set_yscale("log")
    ax.set_xticks(nys)
    ax.set_title(dw_label)
fig.suptitle("Final-cycle Lyapunov gap vs system size — perfect correction", y=1.04)
plt.show()


#### $\Delta_\lambda$ vs cycle for each $N_y$ — perfect correction, DW off vs on

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W2, 2.0), constrained_layout=True)
for ax, cfgs, dw_label in zip(
    axes,
    [PERF_CORR_DW0, PERF_CORR_DW1],
    ["DW off", "DW on"],
):
    for cfg in cfgs:
        ny     = runs[cfg]["summary"]["Ny"]
        color  = COLORS[ny]
        cyc_ax = cycle_axis(cfg)
        mean, err = sample_mean_stderr(runs[cfg]["lyapunov_min_abs"], axis=0)
        ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
        ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
    ax.set_xlabel("cycle")
    ax.set_ylabel(r"$\Delta_\lambda = \min_i |\lambda_i|$")
    ax.set_yscale("log")
    ax.set_title(dw_label)
    ax.legend(frameon=False)
fig.suptitle("Lyapunov gap vs cycle by system size — perfect correction", y=1.04)
plt.show()


## Part 2: Post-Selection


### 2.1 Post-Selection — DW = False


#### Lyapunov spectra histograms at selected cycles — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
_all_vals = np.concatenate([
    runs[c]["spectra"][:, row_cycle(c, cyc) - 1, :].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_bins = np.linspace(_all_vals.min(), _all_vals.max(), 81)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["spectra"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :].ravel()
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\lambda$")
annotate_grid(axes, _cfgs)
fig.suptitle("Lyapunov spectra — post-selection, DW off", y=1.02)
plt.show()


#### Sample-averaged sorted Lyapunov spectrum — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny    = runs[cfg]["summary"]["Ny"]
    color = COLORS[ny]
    n_vec = runs[cfg]["spectra"].shape[2]
    frac  = np.arange(n_vec) / n_vec
    for row, cyc in enumerate(ROW_CYCLES):
        ax        = axes[row, col]
        mean_spec = np.sort(np.mean(runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :], axis=0))
        ax.plot(frac, mean_spec, color=color, lw=1.0)
        if row == 3:
            ax.set_xlabel(r"index $/ n_{\rm vec}$")
annotate_grid(axes, _cfgs)
fig.suptitle("Sample-averaged Lyapunov spectra — post-selection, DW off", y=1.02)
plt.show()


#### Lyapunov gap proxy $\Delta_\lambda$ vs cycle — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["lyapunov_min_abs"], axis=0)
    ax.plot(cyc_ax, mean, color=color,
            label=f"$N_y={ny}$ (S={runs[cfg]['spectra'].shape[0]})")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\min_i |\lambda_i|$")
ax.set_yscale("log")
ax.set_title("Lyapunov gap proxy $\\Delta_\\lambda$ — post-selection, DW off")
ax.legend(frameon=False)
plt.show()


#### Real-space Chern number histograms at selected cycles — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
_all_chern = np.concatenate([
    runs[c]["chern"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(1.0, float(_all_chern.max() - _all_chern.min()))
_bins = np.linspace(float(_all_chern.min()) - _pad, float(_all_chern.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["chern"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["chern"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$C$")
annotate_grid(axes, _cfgs)
fig.suptitle("Real-space Chern number histograms — post-selection, DW off", y=1.02)
plt.show()


#### Sample-averaged Chern number $\langle\mathcal{C}\rangle$ vs cycle — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["chern"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$C$")
ax.set_title("Sample-averaged Chern number — post-selection, DW off")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction histograms at selected cycles — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
_all_fill = np.concatenate([
    runs[c]["filling_fraction"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(0.01, float(_all_fill.max() - _all_fill.min()))
_bins = np.linspace(float(_all_fill.min()) - _pad, float(_all_fill.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["filling_fraction"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["filling_fraction"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
annotate_grid(axes, _cfgs)
fig.suptitle("Filling-fraction histograms — post-selection, DW off", y=1.02)
plt.show()


#### Sample-averaged filling fraction $\nu$ vs cycle — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["filling_fraction"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
ax.set_title("Sample-averaged filling fraction — post-selection, DW off")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction sample-to-sample std vs cycle — DW off, post-selection

In [ ]:
_cfgs = POSTSEL_DW0
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    ff  = runs[cfg]["filling_fraction"]
    std = np.std(ff, axis=0, ddof=1) if ff.shape[0] > 1 else np.zeros(ff.shape[1])
    ax.plot(cyc_ax, std, color=color, label=f"$N_y={ny}$")
ax.set_xlabel("cycle")
ax.set_ylabel(r"std$_{\rm samples}(\nu)$")
ax.set_title("Filling-fraction sample std — post-selection, DW off")
ax.legend(frameon=False)
plt.show()


#### Charge $x$-profile mean vs cycle — DW off, post-selection

In [ ]:
_cfgs   = POSTSEL_DW0
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]   # (S, cycles, n_x)
    mean, _ = sample_mean_stderr(xp, axis=0)
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, mean[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"$\langle N_y^{-1}\sum_y Q_{x,y}\rangle$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile mean — post-selection, DW off", y=1.08)
plt.show()


#### Charge $x$-profile sample-to-sample std vs cycle — DW off, post-selection

In [ ]:
_cfgs   = POSTSEL_DW0
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]
    std    = np.std(xp, axis=0, ddof=1) if xp.shape[0] > 1 else np.zeros(xp.shape[1:])
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, std[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"std$_{\rm samples}(N_y^{-1}\sum_y Q_{x,y})$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile sample std — post-selection, DW off", y=1.08)
plt.show()


### 2.2 Post-Selection — DW = True


#### Lyapunov spectra histograms at selected cycles — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
_all_vals = np.concatenate([
    runs[c]["spectra"][:, row_cycle(c, cyc) - 1, :].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_bins = np.linspace(_all_vals.min(), _all_vals.max(), 81)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["spectra"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :].ravel()
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\lambda$")
annotate_grid(axes, _cfgs)
fig.suptitle("Lyapunov spectra — post-selection, DW on", y=1.02)
plt.show()


#### Sample-averaged sorted Lyapunov spectrum — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny    = runs[cfg]["summary"]["Ny"]
    color = COLORS[ny]
    n_vec = runs[cfg]["spectra"].shape[2]
    frac  = np.arange(n_vec) / n_vec
    for row, cyc in enumerate(ROW_CYCLES):
        ax        = axes[row, col]
        mean_spec = np.sort(np.mean(runs[cfg]["spectra"][:, row_cycle(cfg, cyc) - 1, :], axis=0))
        ax.plot(frac, mean_spec, color=color, lw=1.0)
        if row == 3:
            ax.set_xlabel(r"index $/ n_{\rm vec}$")
annotate_grid(axes, _cfgs)
fig.suptitle("Sample-averaged Lyapunov spectra — post-selection, DW on", y=1.02)
plt.show()


#### Lyapunov gap proxy $\Delta_\lambda$ vs cycle — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["lyapunov_min_abs"], axis=0)
    ax.plot(cyc_ax, mean, color=color,
            label=f"$N_y={ny}$ (S={runs[cfg]['spectra'].shape[0]})")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\min_i |\lambda_i|$")
ax.set_yscale("log")
ax.set_title("Lyapunov gap proxy $\\Delta_\\lambda$ — post-selection, DW on")
ax.legend(frameon=False)
plt.show()


#### Real-space Chern number histograms at selected cycles — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
_all_chern = np.concatenate([
    runs[c]["chern"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(1.0, float(_all_chern.max() - _all_chern.min()))
_bins = np.linspace(float(_all_chern.min()) - _pad, float(_all_chern.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["chern"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["chern"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$C$")
annotate_grid(axes, _cfgs)
fig.suptitle("Real-space Chern number histograms — post-selection, DW on", y=1.02)
plt.show()


#### Sample-averaged Chern number $\langle\mathcal{C}\rangle$ vs cycle — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["chern"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$C$")
ax.set_title("Sample-averaged Chern number — post-selection, DW on")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction histograms at selected cycles — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
_all_fill = np.concatenate([
    runs[c]["filling_fraction"][:, row_cycle(c, cyc) - 1].ravel()
    for c in _cfgs for cyc in ROW_CYCLES
])
_pad  = 0.05 * max(0.01, float(_all_fill.max() - _all_fill.min()))
_bins = np.linspace(float(_all_fill.min()) - _pad, float(_all_fill.max()) + _pad, 13)
fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4), constrained_layout=True)
for col, cfg in enumerate(_cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    n_samp = runs[cfg]["filling_fraction"].shape[0]
    for row, cyc in enumerate(ROW_CYCLES):
        ax   = axes[row, col]
        vals = runs[cfg]["filling_fraction"][:, row_cycle(cfg, cyc) - 1]
        hist_or_vline(ax, vals, _bins, color, n_samp)
        if row == 3:
            ax.set_xlabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
annotate_grid(axes, _cfgs)
fig.suptitle("Filling-fraction histograms — post-selection, DW on", y=1.02)
plt.show()


#### Sample-averaged filling fraction $\nu$ vs cycle — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    mean, err = sample_mean_stderr(runs[cfg]["filling_fraction"], axis=0)
    ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
    ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
ax.set_title("Sample-averaged filling fraction — post-selection, DW on")
ax.legend(frameon=False)
plt.show()


#### Filling-fraction sample-to-sample std vs cycle — DW on, post-selection

In [ ]:
_cfgs = POSTSEL_DW1
fig, ax = plt.subplots(figsize=(W1, 2.0), constrained_layout=True)
for cfg in _cfgs:
    ny     = runs[cfg]["summary"]["Ny"]
    color  = COLORS[ny]
    cyc_ax = cycle_axis(cfg)
    ff  = runs[cfg]["filling_fraction"]
    std = np.std(ff, axis=0, ddof=1) if ff.shape[0] > 1 else np.zeros(ff.shape[1])
    ax.plot(cyc_ax, std, color=color, label=f"$N_y={ny}$")
ax.set_xlabel("cycle")
ax.set_ylabel(r"std$_{\rm samples}(\nu)$")
ax.set_title("Filling-fraction sample std — post-selection, DW on")
ax.legend(frameon=False)
plt.show()


#### Charge $x$-profile mean vs cycle — DW on, post-selection

In [ ]:
_cfgs   = POSTSEL_DW1
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]   # (S, cycles, n_x)
    mean, _ = sample_mean_stderr(xp, axis=0)
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, mean[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"$\langle N_y^{-1}\sum_y Q_{x,y}\rangle$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile mean — post-selection, DW on", y=1.08)
plt.show()


#### Charge $x$-profile sample-to-sample std vs cycle — DW on, post-selection

In [ ]:
_cfgs   = POSTSEL_DW1
_x_show  = np.arange(NX // 2)
_cmap    = mpl.colormaps["tab10"]
_xcolors = {int(x): _cmap(i % 10) for i, x in enumerate(_x_show)}
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.0), sharey=True, constrained_layout=True)
legend_handles = []
for ax, cfg in zip(axes, _cfgs):
    ny     = runs[cfg]["summary"]["Ny"]
    cyc_ax = cycle_axis(cfg)
    xp     = runs[cfg]["x_profile"][:, :, _x_show]
    std    = np.std(xp, axis=0, ddof=1) if xp.shape[0] > 1 else np.zeros(xp.shape[1:])
    dw_loc = runs[cfg]["summary"]["dw_metadata"].get("dw_loc", [])
    for li, x in enumerate(_x_show):
        col = _xcolors[int(x)]
        ln, = ax.plot(cyc_ax, std[:, li], color=col, lw=0.9, label=f"$x={int(x)}$")
        if ax is axes[0]:
            legend_handles.append(ln)
    note = f"DW $x={dw_loc}$" if dw_loc else "no DW"
    ax.text(0.02, 0.97, note, transform=ax.transAxes, ha="left", va="top", fontsize=6)
    ax.set_title(f"$N_y={ny}$")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"std$_{\rm samples}(N_y^{-1}\sum_y Q_{x,y})$")
fig.legend(handles=legend_handles, loc="upper right", ncol=2, frameon=False, fontsize=6,
           bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Charge $x$-profile sample std — post-selection, DW on", y=1.08)
plt.show()


### 2.3 Summary: Lyapunov Gap Scaling (Post-Selection)

Same as §1.3 but for the **post-selection** trajectory ($S=1$ per size).
Error bars are zero (single sample); data points are shown as markers only.


#### Final-cycle $\Delta_\lambda$ vs $N_y$ — post-selection, DW off vs on

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W2, 2.0), constrained_layout=True)
for ax, cfgs, dw_label in zip(
    axes,
    [POSTSEL_DW0, POSTSEL_DW1],
    ["DW off", "DW on"],
):
    nys, means, errs = [], [], []
    for cfg in cfgs:
        ny      = runs[cfg]["summary"]["Ny"]
        final_i = runs[cfg]["summary"]["cycles"] - 1
        gap     = runs[cfg]["lyapunov_min_abs"][:, final_i]
        m, e    = sample_mean_stderr(gap, axis=0)
        nys.append(ny); means.append(float(m)); errs.append(float(e))
    nys = np.array(nys)
    ax.errorbar(nys, means, yerr=errs, fmt="o-", color="tab:blue", capsize=3, ms=5)
    ax.set_xlabel(r"$N_y$")
    ax.set_ylabel(r"$\Delta_\lambda$ (final cycle)")
    ax.set_yscale("log")
    ax.set_xticks(nys)
    ax.set_title(dw_label)
fig.suptitle("Final-cycle Lyapunov gap vs system size — post-selection", y=1.04)
plt.show()


#### $\Delta_\lambda$ vs cycle for each $N_y$ — post-selection, DW off vs on

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W2, 2.0), constrained_layout=True)
for ax, cfgs, dw_label in zip(
    axes,
    [POSTSEL_DW0, POSTSEL_DW1],
    ["DW off", "DW on"],
):
    for cfg in cfgs:
        ny     = runs[cfg]["summary"]["Ny"]
        color  = COLORS[ny]
        cyc_ax = cycle_axis(cfg)
        mean, err = sample_mean_stderr(runs[cfg]["lyapunov_min_abs"], axis=0)
        ax.plot(cyc_ax, mean, color=color, label=f"$N_y={ny}$")
        ax.fill_between(cyc_ax, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
    ax.set_xlabel("cycle")
    ax.set_ylabel(r"$\Delta_\lambda = \min_i |\lambda_i|$")
    ax.set_yscale("log")
    ax.set_title(dw_label)
    ax.legend(frameon=False)
fig.suptitle("Lyapunov gap vs cycle by system size — post-selection", y=1.04)
plt.show()


## Part 3: Min-$|\lambda|$ Eigenvector — Component-Resolved Spatial Structure

For each sample the file `lyapunov_min_abs_vector.npz` stores a single complex vector $v^*\in\mathbb{C}^{N_{\rm layer}}$ ($N_{\rm layer}=2N_xN_y=1600$ for $N_y=40$).  Using the flat index $j=\mu+2x+2N_x y$ we reshape to $v^*[s,y,x,\mu]$, giving the two orbital components $\mu\in\{0,1\}$ directly.  All figures below use $N_y=40$, DW on, for both perfect-correction (PC, $S=25$) and post-selection (PS, $S=1$).

In [ ]:
# Load min-|λ| eigenvectors for Ny=40, DW on, both protocols.
NY40 = 40
cfg_pc = "N20x40_DW1_dwtrunc1_a2-30_nsh1_perfect_correction"
cfg_ps = "N20x40_DW1_dwtrunc1_a2-30_nsh1_postselect"

def load_vec_reshaped(cfg, ny):
    with np.load(CAMPAIGN_ROOT / "runs" / cfg / "lyapunov_min_abs_vector.npz",
                 allow_pickle=False) as d:
        vec = d["lyapunov_min_abs_vector"].copy()   # (S, Nlayer)
    return vec.reshape(vec.shape[0], ny, NX, 2)    # (S, Ny, Nx, 2)  [s,y,x,μ]

v_pc = load_vec_reshaped(cfg_pc, NY40)   # (25, 40, 20, 2)
v_ps = load_vec_reshaped(cfg_ps, NY40)   # ( 1, 40, 20, 2)
print("PC shape:", v_pc.shape, "  PS shape:", v_ps.shape)

#### PC, single sample ($s=0$): $|\psi_\mu(x,y)|^2$ heatmaps — one panel per orbital component

In [ ]:
_s = np.random.randint(25)
_dens = np.abs(v_pc[_s]) ** 2                              # (Ny, Nx, 2)
_dens_norm = _dens / _dens.sum(axis=(0, 1), keepdims=True) # normalise per component

fig, axes = plt.subplots(1, 2, figsize=(W2, 2.6), constrained_layout=True)
for ax, mu in zip(axes, [0, 1]):
    im = ax.imshow(_dens_norm[:, :, mu], origin="lower", aspect="auto",
                   extent=[-0.5, NX - 0.5, -0.5, NY40 - 0.5], cmap="inferno")
    ax.axvline(5,  color="w", lw=0.9, ls="--", alpha=0.8)
    ax.axvline(15, color="w", lw=0.9, ls="--", alpha=0.8)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_title(fr"$\mu={mu}$")
    plt.colorbar(im, ax=ax, label=r"$|\psi_\mu(x,y)|^2 / \sum_{x,y}|\psi_\mu|^2$", pad=0.02)
fig.suptitle(fr"Single sample $s={_s}$: min-$|\lambda|$ density — PC, DW on, $N_y=40$")
fig.savefig(f"../../figs/fig_eigvec_heatmap_pc_single.pdf", dpi=DPI, bbox_inches="tight")
plt.show()

#### PC, sample-averaged: $\langle|\psi_\mu(x,y)|^2\rangle_s$ heatmaps

In [ ]:
_dens_all  = np.abs(v_pc) ** 2                                      # (S, Ny, Nx, 2)
_norm      = _dens_all.sum(axis=(1, 2), keepdims=True)              # (S, 1, 1, 2)
_dens_norm = _dens_all / _norm                                       # (S, Ny, Nx, 2)
_dens_avg  = _dens_norm.mean(axis=0)                                 # (Ny, Nx, 2)

fig, axes = plt.subplots(1, 2, figsize=(W2, 2.6), constrained_layout=True)
for ax, mu in zip(axes, [0, 1]):
    im = ax.imshow(_dens_avg[:, :, mu], origin="lower", aspect="auto",
                   extent=[-0.5, NX - 0.5, -0.5, NY40 - 0.5], cmap="inferno")
    ax.axvline(5,  color="w", lw=0.9, ls="--", alpha=0.8)
    ax.axvline(15, color="w", lw=0.9, ls="--", alpha=0.8)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_title(fr"$\mu={mu}$")
    plt.colorbar(im, ax=ax, label=r"$\langle|\psi_\mu(x,y)|^2 / \sum_{x,y}|\psi_\mu|^2\rangle$", pad=0.02)
fig.suptitle(r"Sample-averaged density — PC, DW on, $N_y=40$")
fig.savefig(f"../../figs/fig_eigvec_heatmap_pc_avg.pdf", dpi=DPI, bbox_inches="tight")
plt.show()

#### PS (single trajectory): $|\psi_\mu(x,y)|^2$ heatmaps

In [ ]:
_dens_ps      = np.abs(v_ps[0]) ** 2                                    # (Ny, Nx, 2)
_dens_ps_norm = _dens_ps / _dens_ps.sum(axis=(0, 1), keepdims=True)     # normalise per component

fig, axes = plt.subplots(1, 2, figsize=(W2, 2.6), constrained_layout=True)
for ax, mu in zip(axes, [0, 1]):
    im = ax.imshow(_dens_ps_norm[:, :, mu], origin="lower", aspect="auto",
                   extent=[-0.5, NX - 0.5, -0.5, NY40 - 0.5], cmap="inferno")
    ax.axvline(5,  color="w", lw=0.9, ls="--", alpha=0.8)
    ax.axvline(15, color="w", lw=0.9, ls="--", alpha=0.8)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_title(fr"$\mu={mu}$")
    plt.colorbar(im, ax=ax, label=r"$|\psi_\mu(x,y)|^2 / \sum_{x,y}|\psi_\mu|^2$", pad=0.02)
fig.suptitle(r"Single trajectory: min-$|\lambda|$ density (per-component norm) — PS, DW on, $N_y=40$")
fig.savefig(f"../../figs/fig_eigvec_heatmap_ps.pdf", dpi=DPI, bbox_inches="tight")
plt.show()

#### PC, single sample ($s=0$): $|\psi_\mu(x_0, y)|^2$ vs $y$ at $x_0=N_x/2$, renormalized

In [ ]:
_y0 = NY40 // 2   # = 20
_x  = np.arange(NX)

_slice_s0 = np.abs(v_pc[0, _y0, :, :]) ** 2       # (Nx, 2)
_slice_s0_sum  = _slice_s0.sum(axis=-1)            # (Nx,)  sum over components
_slice_s0_norm = _slice_s0_sum / _slice_s0_sum.sum()  # normalise

fig, ax = plt.subplots(figsize=(W1, 2.2), constrained_layout=True)
ax.plot(_x, _slice_s0_norm, lw=1.2, color="C0")
ax.axvline(5,  color="tab:green", lw=0.9, ls="--", alpha=0.8, label="DW ($x=5,15$)")
ax.axvline(15, color="tab:green", lw=0.9, ls="--", alpha=0.8)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$\sum_\mu|\psi_\mu(x,y_0)|^2 \;/\; \sum_{x,\mu}|\psi_\mu|^2$")
ax.set_title(fr"Single sample $s=0$, $y_0={_y0}$ — PC, DW on, $N_y=40$")
ax.legend(frameon=False)
fig.savefig(f"../../figs/fig_eigvec_ycut_pc_single.pdf", dpi=DPI, bbox_inches="tight")
plt.show()

#### PC, sample-averaged: $\langle|\psi_\mu(x_0,y)|^2\rangle_s$ vs $y$ at $x_0=N_x/2$, renormalized per sample

In [ ]:
_y0 = NY40 // 2
_x  = np.arange(NX)

_slice_all = np.abs(v_pc[:, _y0, :, :]) ** 2       # (S, Nx, 2)
_slice_sum = _slice_all.sum(axis=-1)                # (S, Nx)  sum over components
_norm      = _slice_sum.sum(axis=1, keepdims=True)  # (S, 1)
_slice_norm = _slice_sum / _norm                    # (S, Nx)
_slice_mean = _slice_norm.mean(axis=0)              # (Nx,)
_slice_err  = _slice_norm.std(axis=0) / np.sqrt(v_pc.shape[0])  # (Nx,)

fig, ax = plt.subplots(figsize=(W1, 2.2), constrained_layout=True)
ax.plot(_x, _slice_mean, lw=1.2, color="C0")
ax.fill_between(_x, _slice_mean - _slice_err, _slice_mean + _slice_err,
                alpha=0.3, color="C0")
ax.axvline(5,  color="tab:green", lw=0.9, ls="--", alpha=0.8, label="DW ($x=5,15$)")
ax.axvline(15, color="tab:green", lw=0.9, ls="--", alpha=0.8)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$\langle\sum_\mu|\psi_\mu(x,y_0)|^2 \;/\; \sum_{x,\mu}|\psi_\mu|^2\rangle$")
ax.set_title(fr"Sample-averaged, $y_0={_y0}$ — PC, DW on, $N_y=40$")
ax.legend(frameon=False)
fig.savefig(f"../../figs/fig_eigvec_ycut_pc_avg.pdf", dpi=DPI, bbox_inches="tight")
plt.show()

#### PS (single trajectory): $|\psi_\mu(x_0,y)|^2$ vs $y$ at $x_0=N_x/2$, renormalized

In [ ]:
_y0 = NY40 // 2
_x  = np.arange(NX)

_slice_ps = np.abs(v_ps[0, _y0, :, :]) ** 2        # (Nx, 2)
_slice_ps_sum  = _slice_ps.sum(axis=-1)             # (Nx,)  sum over components
_slice_ps_norm = _slice_ps_sum / _slice_ps_sum.sum()  # normalise

fig, ax = plt.subplots(figsize=(W1, 2.2), constrained_layout=True)
ax.plot(_x, _slice_ps_norm, lw=1.2, color="C1")
ax.axvline(5,  color="tab:green", lw=0.9, ls="--", alpha=0.8, label="DW ($x=5,15$)")
ax.axvline(15, color="tab:green", lw=0.9, ls="--", alpha=0.8)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$\sum_\mu|\psi_\mu(x,y_0)|^2 \;/\; \sum_{x,\mu}|\psi_\mu|^2$")
ax.set_title(fr"Single trajectory, $y_0={_y0}$ — PS, DW on, $N_y=40$")
ax.legend(frameon=False)
fig.savefig(f"../../figs/fig_eigvec_ycut_ps.pdf", dpi=DPI, bbox_inches="tight")
plt.show()